In [ ]:
import os
import pandas as pd

### Load and reformat BioPlex PPI for n2v input

In [ ]:
### Convert BioPlex 3 old gene symbol to HGNC
hgnc = pd.read_csv("../../data/HGNC/hgnc_complete_set.txt", sep="\t", low_memory=False)
hgnc = hgnc[hgnc["status"] == "Approved"]
approved_symbols = set(hgnc["symbol"])

with_entrez = hgnc.dropna(subset=["entrez_id"])
entrez2symbol = dict(zip(with_entrez["entrez_id"].astype(int), with_entrez["symbol"]))

# retired symbol -> current symbol, keeping only names that resolve to a single approved gene
old2symbol = {}
for column in ["prev_symbol", "alias_symbol"]:
    for symbol, blob in zip(hgnc["symbol"], hgnc[column]):
        if not isinstance(blob, str):
            continue
        for old in blob.split("|"):
            old = old.strip()
            if old == "" or old in approved_symbols:
                continue
            old2symbol.setdefault(old, set()).add(symbol)
old2symbol = {old: next(iter(v)) for old, v in old2symbol.items() if len(v) == 1}

In [ ]:
def bioplex_to_ppi(
    infile,
    outfile,
):
    df = pd.read_csv(infile, sep="\t")

    ppi = pd.DataFrame({
        "GeneA": df["GeneA"].map(entrez2symbol),
        "GeneB": df["GeneB"].map(entrez2symbol),
        "Score": df["pInt"],
    })
    
    # Prioritize to use HGNC if possible, then old symbol for the rest
    ppi["GeneA"] = ppi["GeneA"].fillna(df["SymbolA"].map(old2symbol)).fillna(df["SymbolA"])
    ppi["GeneB"] = ppi["GeneB"].fillna(df["SymbolB"].map(old2symbol)).fillna(df["SymbolB"])
    n_renamed = ((ppi["GeneA"] != df["SymbolA"]) | (ppi["GeneB"] != df["SymbolB"])).sum()

    # Merges nodes that BioPlex listed twice under different symbols
    geneA = ppi[["GeneA", "GeneB"]].min(axis=1)
    geneB = ppi[["GeneA", "GeneB"]].max(axis=1)
    ppi["GeneA"] = geneA
    ppi["GeneB"] = geneB
    n_self = (ppi["GeneA"] == ppi["GeneB"]).sum()
    ppi = ppi[ppi["GeneA"] != ppi["GeneB"]]
    n_merged = ppi.duplicated(subset=["GeneA", "GeneB"]).sum()
    ppi = ppi.groupby(["GeneA", "GeneB"], as_index=False)["Score"].mean()

    ppi.to_csv(outfile, sep="\t", index=False, header=True)
    return

In [ ]:
os.makedirs("../../result/HEK293T_HCT116/input/", exist_ok=True)

bioplex_to_ppi(
    "../../data/BioPlex/BioPlex_293T_Network_10K_Dec_2019.tsv",
    "../../result/HEK293T_HCT116/input/BioPlex-HEK293T.ppi"
)

bioplex_to_ppi(
    "../../data/BioPlex/BioPlex_HCT116_Network_5.5K_Dec_2019.tsv",
    "../../result/HEK293T_HCT116/input/BioPlex-HCT116.ppi"
)